# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [4]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [5]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [6]:
openai = OpenAI(api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1")

In [7]:
# Some lists!

checklist = []
completed = []

In [9]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [10]:
get_checklist_report()

''

In [11]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [12]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [13]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [14]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [15]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [16]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [17]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [18]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [19]:
def loop(messages):
    response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [20]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [21]:
checklist, completed = [], []
loop(messages)

Checklist #1: Estimate the distance between Boston and New York for train travel.
Checklist #2: Determine the start times and speeds of each train.
Checklist #3: Set up the equation for meeting distance based on relative speed.
Checklist #4: Solve for the meeting time and convert to clock time.
Checklist #5: Provide the final answer in a clear statement.

The rail distance between Boston and New York is about 215 miles (≈ 210‑220 mi). I’ll use 215 miles as a reasonable
estimate for the calculation.

Checklist #1: Estimate the distance between Boston and New York for train travel.
Checklist #2: Determine the start times and speeds of each train.
Checklist #3: Set up the equation for meeting distance based on relative speed.
Checklist #4: Solve for the meeting time and convert to clock time.
Checklist #5: Provide the final answer in a clear statement.

Train A (Boston → NY): departs 2:00 pm, speed 60 mph.
Train B (NY → Boston): departs 3:00 pm, speed 80 mph.
Thus from 3:00 pm onward they are moving toward each other with a combined speed of 60 + 80 = 140 mph.

Checklist #1: Estimate the distance between Boston and New York for train travel.
Checklist #2: Determine the start times and speeds of each train.
Checklist #3: Set up the equation for meeting distance based on relative speed.
Checklist #4: Solve for the meeting time and convert to clock time.
Checklist #5: Provide the final answer in a clear statement.

After 3:00 pm, the remaining separation is 215 mi − 60 mi = 155 mi. With a closing speed of 140 mph, the time to 
meet is t = 155 / 140 ≈ 1.107 h ≈ 1 h 6.4 min, i.e., about 1 hour 6 minutes 24 seconds. Adding this to 3:00 pm 
gives a meeting time of roughly **4:06 pm** (plus a few seconds).

Checklist #1: Estimate the distance between Boston and New York for train travel.
Checklist #2: Determine the start times and speeds of each train.
Checklist #3: Set up the equation for meeting distance based on relative speed.
Checklist #4: Solve for the meeting time and convert to clock time.
Checklist #5: Provide the final answer in a clear statement.

**Answer:**  
Assuming the rail distance between Boston and New York is about **215 miles**, the two trains will meet at 
approximately **4:06 pm** (a few seconds later).  

*Explanation:*  
- Train A leaves Boston at 2:00 pm, traveling 60 mph. By 3:00 pm it has covered 60 mi.  
- At 3:00 pm the remaining distance is 215 mi − 60 mi = 155 mi.  
- Train B departs New York at 3:00 pm, traveling 80 mph toward Boston.  
- The trains now approach each other with a combined speed of 60 + 80 = 140 mph.  
- Time to close the 155 mi gap: 155 mi ÷ 140 mph ≈ 1.107 h ≈ 1 h 6 min 24 s.  
- Adding this to 3:00 pm gives a meeting time of roughly **4:06 pm**.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>